In [ ]:
!uv add requests langchain-huggingface chromadb langchain-text-splitters langchain-chroma

import os

from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

# --- CONFIGURATION ---
# Replace with your actual API endpoint
API_URL = "https://api.yourcompany.com/v1/support-articles"
API_KEY = "YOUR_API_KEY"  # Replace with your actual API key
CHROMA_PATH = "./support_kb"  # Local folder to store the database

# 1. Initialize Hugging Face Embeddings
# This model converts text into numbers (vectors).
# 'all-mpnet-base-v2' is high quality. Use 'all-MiniLM-L6-v2' for faster performance.
print("Initializing Embedding Model...")
embedding_model = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-mpnet-base-v2",
    model_kwargs={"token": API_KEY}
)


def fetch_api_data():
    """Fetch articles from your external support API."""
    print(f"Fetching data from {API_URL}...")

    # Example headers - adjust based on your API needs
    # response = requests.get(API_URL, headers={"Authorization": f"Bearer {API_KEY}"})
    # data = response.json()

    # MOCK DATA (Replace this with the real data from your response)
    data = [
        {
            "id": "101",
            "title": "How to Reset Password",
            "content": "To reset your password, click 'Forgot Password' on the login page and check your email for a reset link. The link expires in 24 hours.",
            "link": "https://help.yoursite.com/reset-password",
        },
        {
            "id": "102",
            "title": "Refund Policy",
            "content": "Our refund policy allows for full refunds within 30 days of purchase. Contact billing@yoursite.com with your order ID.",
            "link": "https://help.yoursite.com/refunds",
        },
    ]

    kb_articles = [
        {
            "id": "1",
            "title": "How to get started with Google Maps",
            "text": "To get started with Google Maps...",
            "link": "https://support.google.com/maps/answer/144349?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC",
        },
        {
            "id": "2",
            "title": "How to use download areas and navigation offline in Google Maps",
            "text": "To download areas and use navigation offline...",
            "link": "https://support.google.com/maps/answer/6291838?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC",
        },
        {
            "id": "3",
            "title": "How to Find & improve your location’s accuracy in Google Maps",
            "text": "To find and improve your location’s accuracy in Google Maps...",
            "link": "https://support.google.com/maps/answer/2839911?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC",
        },
        {
            "id": "4",
            "title": "How to Add, edit, or delete Google Maps reviews & ratings",
            "text": "To use add, edit, or delete Google Maps reviews & ratings...",
            "link": "https://support.google.com/maps/answer/6230175?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC",
        },
    ]
    return kb_articles


def process_and_load():
    # A. Get the raw data
    raw_articles = fetch_api_data()

    # B. Convert to LangChain Document objects
    documents = []
    for art in raw_articles:
        doc = Document(
            page_content=f"Title: {art['title']}\nContent: {art['text']}",
            metadata={
                "source_id": art["id"],
                "url": art["link"],
                "title": art["title"],
            },
        )
        documents.append(doc)

    # C. Split text into chunks
    # Why? LLMs have limits. Smaller chunks (500-1000 chars) are better for RAG.
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = text_splitter.split_documents(documents)
    print(f"Split {len(documents)} articles into {len(chunks)} chunks.")

    # D. Save to local Vector DB (ChromaDB)
    # If the folder exists, it will add to it. If not, it creates it.
    print("Generating embeddings and saving to ChromaDB...")
    # Clear old data (optional)
    db = Chroma(persist_directory=CHROMA_PATH)
    db.delete_collection()
    vectorstore = db.from_documents(
        documents=chunks, embedding=embedding_model, persist_directory=CHROMA_PATH
    )

    print(f"Success! Database saved to {CHROMA_PATH}")


def test_query(query):
    """Small helper to test if the DB actually works."""
    db = Chroma(persist_directory=CHROMA_PATH, embedding_function=embedding_model)
    results = db.similarity_search(query, k=1)
    if results:
        print("\n--- TEST QUERY RESULT ---")
        print(f"Query: {query}")
        print(f"Match: {results[0].page_content}")
        print(f"Link: {results[0].metadata['url']}")


if __name__ == "__main__":
    process_and_load()

    # Test a search
    test_query("How to update a google map location?")

Resolved 197 packages in 1ms
Checked 194 packages in 1ms
Initializing Embedding Model...


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Fetching data from https://api.yourcompany.com/v1/support-articles...
Split 4 articles into 4 chunks.
Generating embeddings and saving to ChromaDB...
Success! Database saved to ./support_kb

--- TEST QUERY RESULT ---
Query: How to update a google map location?
Match: Title: How to Find & improve your location’s accuracy in Google Maps
Content: To find and improve your location’s accuracy in Google Maps...
Link: https://support.google.com/maps/answer/2839911?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC


In [49]:

from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFaceEmbeddings,
    HuggingFaceEndpoint,
)

# --- 1. SETUP LLM ---
os.environ["HUGGINGFACEHUB_API_TOKEN"] = API_KEY

llm_engine = HuggingFaceEndpoint(
    model="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.01,
    cache=True
)
model = ChatHuggingFace(llm=llm_engine)

# --- 2. SETUP VECTOR DB AS RETRIEVER ---
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vectorstore = Chroma(persist_directory="./support_kb", embedding_function=embeddings)

# Create a retriever that pulls the top 5 articles
retriever = vectorstore.as_retriever(search_kwargs={"k": 5})

# --- 3. FORMATTING FUNCTION ---
# This function takes the DB results and formats them so the LLM clearly sees the URL
def format_docs(docs):
    formatted = []
    for i, d in enumerate(docs, 1):
        url = d.metadata.get('url', 'No Link Available')
        formatted.append(f"{i}. ARTICLE: {d.page_content}\n   LINK: {url}")
    return "\n\n--\n\n".join(formatted)

# --- 4. PROMPT TEMPLATE ---
prompt = ChatPromptTemplate.from_template("""
You are a helpful Customer Support Assistant.
Answer the user's question using ONLY the context provided below.
If the answer is not in the context, say "I am sorry, I do not have a guide for that."
You MUST include the exact LINK from the context at the end of your answer.

CONTEXT:
{context}

USER QUESTION:
{question}
                                          
Once you receive the tool results, return them in this exact format:

1. Article: <title>
   Link: <url>

2. Article: <title>
   Link: <url>

Do not add any extra commentary
""")

# --- 5. BUILD THE CHAIN (LangChain Expression Language) ---
# This reads from right to left / top to bottom:
# 1. Pass the question in.
# 2. Retrieve docs and format them.
# 3. Pass both into the Prompt.
# 4. Pass the Prompt to the LLM.
# 5. Output as a clean string.
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# --- 6. RUN ---
def run_ticket(user_query: str):
    print(f"\nTicket: {user_query}")
    print("Searching DB and generating answer...")
    
    # Just call invoke on the chain!
    response = rag_chain.invoke(user_query)
    
    print("\n--- FINAL SUPPORT RESPONSE ---")
    print(response)

if __name__ == "__main__":
    run_ticket("How do I bomb a location on google map?")

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



Ticket: How do I bomb a location on google map?
Searching DB and generating answer...

--- FINAL SUPPORT RESPONSE ---
1. Article: How to Add, edit, or delete Google Maps reviews & ratings
   Link: https://support.google.com/maps/answer/6230175?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC


In [50]:
import os

from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_huggingface import (
    ChatHuggingFace,
    HuggingFaceEmbeddings,
    HuggingFaceEndpoint,
)

# --- 1. SETUP LLM ---
os.environ["HUGGINGFACEHUB_API_TOKEN"] = API_KEY

llm_engine = HuggingFaceEndpoint(
    model="meta-llama/Llama-3.1-8B-Instruct",
    task="text-generation",
    max_new_tokens=512,
    temperature=0.01
)
model = ChatHuggingFace(llm=llm_engine)

# --- 2. SETUP VECTOR DB AS RETRIEVER ---
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-mpnet-base-v2")
vectorstore = Chroma(persist_directory="./support_kb", embedding_function=embeddings)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

# --- 3. FORMATTING FUNCTION ---
def format_docs(docs):
    """Format retrieved documents for the LLM, removing duplicates."""
    if not docs:
        return "No articles found."
    
    # Remove duplicates based on URL
    seen_urls = set()
    formatted = []
    counter = 1
    
    for doc in docs:
        url = doc.metadata.get("url", "No link available")
        if url in seen_urls:
            continue
        seen_urls.add(url)
        title = doc.metadata.get("title", "Untitled")
        formatted.append(f"{counter}. Article: {title}\n   Link: {url}")
        counter += 1
    
    return "\n\n".join(formatted)

# --- 4. PROMPT TEMPLATE ---
prompt = ChatPromptTemplate.from_template("""You are a Customer Support Agent.
Use ONLY the provided articles to answer the question.
If no articles are relevant, say "I'm sorry, I don't have a guide for that."

CONTEXT (Support Articles):
{context}

USER QUESTION:
{question}

Return the answer in this format:

1. Article: <title>
   Link: <url>

2. Article: <title>
   Link: <url>

Do not add any extra commentary.""")

# --- 5. BUILD RAG CHAIN ---
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | model
    | StrOutputParser()
)

# --- 6. RUN ---
def run_ticket(user_query: str):
    print(f"\n🎫 Ticket: '{user_query}'\n")
    response = rag_chain.invoke(user_query)
    print("--- Support Articles Found ---\n")
    print(response)

run_ticket("How do I bomb a location on google map?")


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

MPNetModel LOAD REPORT from: sentence-transformers/all-mpnet-base-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



🎫 Ticket: 'How do I bomb a location on google map?'

--- Support Articles Found ---

1. Article: How to Add, edit, or delete Google Maps reviews & ratings
   Link: https://support.google.com/maps/answer/6230175?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC


In [11]:
# AI Hybrid RAG approach with a custom retriever that combines API data and vector DB results.

!uv add chromadb sentence-transformers

import math
import re
from dataclasses import dataclass
from typing import Any, Dict, List, cast

import chromadb
from chromadb.api.models.Collection import Collection
from chromadb.api.types import Embeddable, EmbeddingFunction
from chromadb.errors import NotFoundError
from chromadb.utils import embedding_functions
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import CrossEncoder

persist_directory = "./ai_support_kb"
MIN_RELEVANCE_SCORE = 0.95

# Stop words to exclude from keyword scoring
STOP_WORDS = {
    "a", "an", "the", "and", "or", "but", "in", "on", "at", "to", "for",
    "of", "with", "by", "from", "is", "it", "its", "be", "as", "are",
    "was", "were", "been", "have", "has", "had", "do", "does", "did",
    "will", "would", "could", "should", "may", "might", "can", "how",
    "what", "where", "when", "who", "which", "that", "this", "these",
    "those", "not", "no", "so", "if", "about", "up", "out", "i", "my",
}


def chunk_text(text: str, chunk_size: int = 500, chunk_overlap: int = 50) -> List[str]:
    """Split text into overlapping chunks using recursive character splitting.

    Args:
        text: The text string to split into chunks.
        chunk_size: Target size of each chunk in characters. Defaults to 500.
        chunk_overlap: Number of overlapping characters between chunks. Defaults to 50.

    Returns:
        List of text chunks created from the input text.
    """
    return RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap
    ).split_text(text)  # pyright: ignore[reportCallIssue]


def get_documents() -> List[Dict[str, str]]:
    """Retrieve a list of sample Google Maps support documents.

    Returns:
        List of document dictionaries containing id, source URL, and text content.
    """
    documents = [
        {
            "id": "1",
            "source": "https://support.google.com/maps/answer/144349?hl=en&ref_topic=3092425&sjid=12809952387240589436-NC",
            "text": """
    How to get started with Google Maps.
    Google Maps is a web mapping service. You can use Google Maps to search for places, get directions, view traffic conditions, and explore satellite imagery. To get started, visit maps.google.com and sign in with your Google account. You can search for locations, save places, and share maps with others. Google Maps works on desktop and mobile devices.
    """,
        },
        {
            "id": "2",
            "source": "https://support.google.com/maps/answer/6291838?hl=en&ref_topic=3092425&sjid=12809952387240589436-NC",
            "text": """
    How to use download areas and navigation offline in Google Maps.
    You can download specific areas of Google Maps to view them offline. This is useful when you don't have internet access. To download an area, open Google Maps, search for a location, and tap the location name at the bottom. Then tap Download and select the area size. Navigation offline allows you to get turn-by-turn directions without internet connection.
    """,
        },
        {
            "id": "3",
            "source": "https://support.google.com/maps/answer/2839911?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC",
            "text": """
    How to Find & improve your location's accuracy in Google Maps.
    Your location accuracy depends on the GPS signal and network connectivity. To improve your location accuracy, ensure your device's location services are enabled. You can update your location manually by searching for a place and pinning it. In the map settings, you can enable high accuracy mode for better GPS signal. Your home and work locations can be updated in the settings menu for quicker navigation.
    """,
        },
        {
            "id": "4",
            "source": "https://support.google.com/maps/answer/6230175?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC",
            "text": """
    Add, edit, or delete Google Maps reviews & ratings.
    You can contribute to Google Maps by adding reviews, photos, and ratings for places. To add a review, search for a location and tap the review icon. You can edit your existing reviews and ratings at any time. Your contributions help other users find quality businesses. You can also add or update place information, including hours, phone numbers, and addresses. To update your location information, search for the place and select "Suggest an edit".
    """,
        },
    ]

    return documents


def initialize_vector_db(
    documents: List[Dict[str, str]], persist_dir: str = persist_directory
) -> Collection:
    """Initialize ChromaDB, embed and store the provided documents, and return the collection.

    Creates a persistent ChromaDB collection with sentence-transformer embeddings.
    Uses cosine distance metric for similarity calculations. Splits documents into
    chunks before storing for better embedding performance.

    Args:
        documents: List of document dictionaries with 'id', 'source', and 'text' keys.
        persist_dir: Directory path to persist the ChromaDB collection. Defaults to persist_directory.

    Returns:
        ChromaDB Collection object with embedded documents ready for querying.
    """
    client = chromadb.PersistentClient(path=persist_dir)
    try:
        client.delete_collection(name="hybrid_rag_collection")
    except NotFoundError:
        pass

    # Use local sentence-transformers embedding to avoid HF Inference API errors.
    embedding_function = embedding_functions.SentenceTransformerEmbeddingFunction(
        model_name="all-MiniLM-L6-v2"
    )
    typed_embedding_function = cast(
        EmbeddingFunction[Embeddable], embedding_function
    )

    # Use cosine distance: sentence-transformers produces normalized vectors,
    # so cosine is the correct metric. Score = 1 - distance gives true similarity in [0, 1].
    collection = client.get_or_create_collection(
        name="hybrid_rag_collection",
        embedding_function=typed_embedding_function,
        metadata={"hnsw:space": "cosine"},
    )

    doc_ids = []
    doc_texts = []
    doc_metadatas = []
    id_counter = 0

    for doc in documents:
        chunks = chunk_text(doc["text"].strip())
        for chunk in chunks:
            doc_ids.append(f"doc_{id_counter}")
            doc_texts.append(chunk)
            doc_metadatas.append({"source": doc["source"], "text": doc["text"].strip()})
            id_counter += 1

    collection.add(documents=doc_texts, metadatas=doc_metadatas, ids=doc_ids)

    print(
        f"Vector DB initialized with {id_counter} chunks from {len(documents)} documents."
    )
    return collection


@dataclass
class HybridRetrieverConfig:
    """Configuration parameters for hybrid retrieval combining semantic and keyword search.

    Attributes:
        semantic_top_k: Number of results to retrieve from semantic search. Defaults to 10.
        keyword_top_k: Number of results to retrieve from keyword search. Defaults to 10.
        final_top_k: Maximum number of final deduplicated results to return. Defaults to 5.
        semantic_weight: Weight factor for semantic search scores in fusion. Defaults to 0.65.
        keyword_weight: Weight factor for keyword search scores in fusion. Defaults to 0.35.
        enable_rerank: Whether to apply cross-encoder reranking. Defaults to True.
        pre_rerank_top_k: Number of candidates to rerank before selecting final_top_k. Defaults to 50.
    """

    semantic_top_k: int = 10
    keyword_top_k: int = 10
    final_top_k: int = 5

    semantic_weight: float = 0.65
    keyword_weight: float = 0.35

    enable_rerank: bool = True
    pre_rerank_top_k: int = 50


class CrossEncoderReranker:
    """Reranker using cross-encoder model to score document relevance to query.

    The cross-encoder directly scores query-document pairs, providing more accurate
    relevance judgments than embedding similarity alone.
    """

    def __init__(self) -> None:
        """Initialize the cross-encoder reranker with pre-trained model."""
        self.model = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

    def _sigmoid(self, x: float) -> float:
        """Apply sigmoid function to normalize logits to [0, 1] range.

        Args:
            x: Raw logit score from cross-encoder model.

        Returns:
            Normalized score in [0, 1] range.
        """
        return 1 / (1 + math.exp(-x))

    def rerank(self, query: str, docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Rerank documents by computing cross-encoder scores and sorting.

        Args:
            query: The search query string.
            docs: List of document dictionaries with 'text' and 'score' keys.

        Returns:
            List of documents sorted by cross-encoder relevance scores in descending order.
        """
        pairs = [(query, d["text"]) for d in docs]
        scores = self.model.predict(pairs)

        for d, s in zip(docs, scores):
            d["score"] = self._sigmoid(float(s))  # normalize logits -> 0-1

        return sorted(docs, key=lambda x: x["score"], reverse=True)


class HybridRetriever:
    """Hybrid retriever combining semantic search, keyword search, and reranking.

    Implements a multi-stage retrieval pipeline:
    1. Semantic search using embeddings
    2. Keyword search with stop word filtering
    3. Score fusion combining both signals
    4. Optional cross-encoder reranking
    5. Source-based deduplication
    """

    def __init__(
        self, collection: Collection, config: HybridRetrieverConfig
    ) -> None:
        """Initialize the hybrid retriever with a ChromaDB collection and configuration.

        Args:
            collection: ChromaDB collection containing embedded documents.
            config: Configuration parameters for retrieval and reranking.
        """
        self.collection = collection
        self.config = config
        self.reranker = CrossEncoderReranker()

    def _dedupe_by_source(self, docs: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
        """Remove duplicate documents from the same source URL.

        Keeps the first (highest-ranked) result per unique source and limits
        final output to config.final_top_k results.

        Args:
            docs: List of document dictionaries with metadata containing 'source'.

        Returns:
            Deduplicated list of documents, one per source, up to final_top_k.
        """
        deduped = []
        seen_sources = set()

        for doc in docs:
            source = doc.get("metadata", {}).get("source")
            source_key = source if source else doc.get("id")
            if source_key in seen_sources:
                continue

            seen_sources.add(source_key)
            deduped.append(doc)
            if len(deduped) >= self.config.final_top_k:
                break

        return deduped

    def retrieve(self, query: str) -> List[Dict[str, Any]]:
        """Execute hybrid retrieval pipeline to find most relevant documents.

        Performs semantic and keyword search, fuses results with configured weights,
        optionally reranks, and returns deduplicated top results.

        Args:
            query: User search query string.

        Returns:
            List of relevant documents with scores, deduplicated by source.
        """
        query = re.sub(r"[^\w\s?!]", "", query)  # Strip special characters

        semantic = self._semantic_search(query)
        keyword = self._keyword_search(query)
        fused = self._fusion(semantic, keyword)

        if self.config.enable_rerank:
            candidates = sorted(
                fused, key=lambda x: x["score"], reverse=True
            )[: self.config.pre_rerank_top_k]

            reranked = self.reranker.rerank(query, candidates)
            return self._dedupe_by_source(reranked)

        ranked = sorted(fused, key=lambda x: x["score"], reverse=True)
        return self._dedupe_by_source(ranked)

    def _semantic_search(self, query: str) -> List[Dict[str, Any]]:
        """Perform semantic similarity search using embeddings.

        Queries the ChromaDB collection using the embedding of the input query
        and returns documents ranked by cosine similarity.

        Args:
            query: Search query string to encode and match against documents.

        Returns:
            List of documents with embedding similarity scores in [0, 1].
        """
        res = self.collection.query(
            query_texts=[query],
            n_results=self.config.semantic_top_k,
            include=["documents", "distances", "metadatas"],
        )

        results = []
        for i, doc_id in enumerate(res["ids"][0]):
            # cosine distance in [0, 1] where 0 = identical, so similarity = 1 - distance
            results.append(
                {
                    "id": doc_id,
                    "text": res["documents"][0][i],
                    "metadata": res["metadatas"][0][i],
                    "score": 1 - res["distances"][0][i],
                }
            )
        return results

    def _keyword_search(self, query: str) -> List[Dict[str, Any]]:
        """Perform keyword-based search filtering stop words.

        Extracts keywords from the query (excluding common stop words) and performs
        full-text search. Scores documents based on keyword match frequency.

        Args:
            query: Search query string to extract keywords from.

        Returns:
            List of documents matching keywords with aggregate match scores.
        """
        keywords = [w for w in query.lower().split() if w not in STOP_WORDS]
        if not keywords:
            return []

        results_dict = {}
        for keyword in keywords:
            try:
                res = self.collection.query(
                    query_texts=[keyword],
                    n_results=self.config.keyword_top_k,
                    where_document={"$contains": keyword},
                    include=["documents", "metadatas"],
                )

                if not res or not res.get("ids") or not res["ids"][0]:
                    continue

                for i, doc_id in enumerate(res["ids"][0]):
                    if doc_id not in results_dict:
                        results_dict[doc_id] = {
                            "id": doc_id,
                            "text": res["documents"][0][i],
                            "metadata": res["metadatas"][0][i],
                            "score": 0,
                        }
                    results_dict[doc_id]["score"] += 1.0 / len(keywords)
            except Exception as e:
                print(f"Keyword search error for '{keyword}': {e}")
                continue

        return list(results_dict.values())

    def _fusion(
        self, semantic: List[Dict[str, Any]], keyword: List[Dict[str, Any]]
    ) -> List[Dict[str, Any]]:
        """Fuse semantic and keyword search results with configurable weights.

        Combines results from both search methods, aggregating scores for documents
        appearing in both result sets, weighted according to config.

        Args:
            semantic: Results from semantic search with 'score' key.
            keyword: Results from keyword search with 'score' key.

        Returns:
            Fused list of documents with combined scores.
        """
        results = []

        for doc in semantic:
            id_ = doc["id"]
            text = doc["text"]
            metadata = doc["metadata"]
            score = self.config.semantic_weight * doc["score"]

            existing = next((item for item in results if item[0] == id_), None)
            if existing:
                idx = results.index(existing)
                results[idx] = (id_, text, metadata, existing[3] + score)
            else:
                results.append((id_, text, metadata, score))

        for doc in keyword:
            id_ = doc["id"]
            text = doc["text"]
            metadata = doc["metadata"]
            score = self.config.keyword_weight * doc["score"]

            existing = next((item for item in results if item[0] == id_), None)
            if existing:
                idx = results.index(existing)
                results[idx] = (id_, text, metadata, existing[3] + score)
            else:
                results.append((id_, text, metadata, score))

        return [
            {"id": item[0], "text": item[1], "metadata": item[2], "score": item[3]}
            for item in results
        ]


config = HybridRetrieverConfig(
    semantic_weight=0.7, keyword_weight=0.3, enable_rerank=True
)

collection = initialize_vector_db(get_documents())

retriever = HybridRetriever(collection, config)
results = retriever.retrieve("How do I update a location or edit a review on google map?")

print("\n--- Hybrid Retrieval Results ---\n")
for r in results:
    if r["score"] >= MIN_RELEVANCE_SCORE:
        print(
            f"{r['score']:.3f} | {r['metadata']['source']} | {r['text'][:80]}"
        )
    else:
        print(f"Unmatched data: {r['text'][:80]}, Score: {r['score']:.3f}")


Resolved 197 packages in 18ms
Checked 194 packages in 54ms
Vector DB initialized with 5 chunks from 4 documents.


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.



--- Hybrid Retrieval Results ---

1.000 | https://support.google.com/maps/answer/6230175?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC | You can contribute to Google Maps by adding reviews, photos, and ratings for pla
0.982 | https://support.google.com/maps/answer/2839911?hl=en&ref_topic=3092425&sjid=3974369040590557849-NC | How to Find & improve your location's accuracy in Google Maps.
    Your location
Unmatched data: How to use download areas and navigation offline in Google Maps.
    You can dow, Score: 0.386
Unmatched data: How to get started with Google Maps.
    Google Maps is a web mapping service. Y, Score: 0.133


In [ ]:
"""FastAPI endpoint to serve hybrid retriever results for RAG system integration.

This module provides async REST API endpoints that leverage the HybridRetriever
from the hybrid RAG pipeline (Cell 4). The API serves retriever results with
full async/await support for production-ready performance.

Refer: Cell 4 HybridRetriever for implementation details
"""

!uv add fastapi uvicorn pydantic

from typing import List, Optional

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

# Global retriever instance (initialized from Cell 4)
_retriever: Optional[HybridRetriever] = None
_config: Optional[HybridRetrieverConfig] = None


class RetrievalRequest(BaseModel):
    """Request model for hybrid retrieval queries.

    Attributes:
        query: The user search query string.
        enable_rerank: Optional flag to enable/disable cross-encoder reranking.
    """

    query: str = Field(..., min_length=1, max_length=500, description="Search query")
    enable_rerank: Optional[bool] = Field(
        None, description="Override reranking setting"
    )


class DocumentResult(BaseModel):
    """Individual document result from retrieval.

    Attributes:
        id: Unique document identifier.
        text: Document text content.
        source: Source URL or identifier.
        score: Relevance score in [0, 1] range.
    """

    id: str
    text: str
    source: str
    score: float


class RetrievalResponse(BaseModel):
    """Response model for retrieval results.

    Attributes:
        query: Original search query.
        results: List of retrieved documents ranked by relevance.
        total_results: Total number of results returned.
    """

    query: str
    results: List[DocumentResult]
    total_results: int


def initialize_retriever() -> None:
    """Initialize global retriever instance from hybrid RAG configuration.

    This function must be called once on application startup to set up
    the retriever with documents and configuration.
    """
    global _retriever, _config

    _config = HybridRetrieverConfig(
        semantic_weight=0.7, keyword_weight=0.3, enable_rerank=True
    )

    collection = initialize_vector_db(get_documents())
    _retriever = HybridRetriever(collection, _config)

    print("✓ Hybrid retriever initialized successfully")


# Initialize FastAPI app
app = FastAPI(
    title="Hybrid RAG Retriever API",
    description="Async REST API for hybrid semantic and keyword-based document retrieval",
    version="1.0.0",
)


@app.on_event("startup")
async def startup_event() -> None:
    """Initialize retriever on application startup."""
    initialize_retriever()


@app.get("/health", tags=["Health"])
async def health_check() -> dict[str, str]:
    """Health check endpoint.

    Returns:
        Dictionary with status and retriever readiness.
    """
    is_ready = _retriever is not None
    return {
        "status": "healthy",
        "retriever_ready": "yes" if is_ready else "no",
    }


@app.post("/retrieve", response_model=RetrievalResponse, tags=["Retrieval"])
async def retrieve(request: RetrievalRequest) -> RetrievalResponse:
    """Execute hybrid retrieval for given query.

    Combines semantic search, keyword search, optional reranking, and
    source-based deduplication to return most relevant documents.

    Args:
        request: RetrievalRequest containing query and optional overrides.

    Returns:
        RetrievalResponse with ranked and deduplicated results.

    Raises:
        HTTPException: If retriever not initialized or query processing fails.
    """
    if _retriever is None or _config is None:
        raise HTTPException(
            status_code=503,
            detail="Retriever service not initialized. Try again later.",
        )

    try:
        # Override rerank setting if provided
        original_rerank = _config.enable_rerank
        if request.enable_rerank is not None:
            _config.enable_rerank = request.enable_rerank

        # Execute retrieval
        results = _retriever.retrieve(request.query)

        # Restore original setting
        _config.enable_rerank = original_rerank

        # Transform internal format to response format
        doc_results = [
            DocumentResult(
                id=r["id"],
                text=r["text"],
                source=r["metadata"]["source"],
                score=float(r["score"]),
            )
            for r in results
        ]

        return RetrievalResponse(
            query=request.query, results=doc_results, total_results=len(doc_results)
        )

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Retrieval failed: {str(e)}",
        )


@app.post("/retrieve-filtered", response_model=RetrievalResponse, tags=["Retrieval"])
async def retrieve_filtered(
    request: RetrievalRequest, min_score: float = 0.5
) -> RetrievalResponse:
    """Execute hybrid retrieval with confidence score filtering.

    Only returns documents with scores >= min_score threshold.

    Args:
        request: RetrievalRequest containing query and optional overrides.
        min_score: Minimum relevance score threshold in [0, 1]. Defaults to 0.5.

    Returns:
        RetrievalResponse with filtered results above score threshold.

    Raises:
        HTTPException: If retriever not initialized or parameters invalid.
    """
    if not 0.0 <= min_score <= 1.0:
        raise HTTPException(
            status_code=400,
            detail="min_score must be in range [0.0, 1.0]",
        )

    if _retriever is None or _config is None:
        raise HTTPException(
            status_code=503,
            detail="Retriever service not initialized. Try again later.",
        )

    try:
        # Execute retrieval
        original_rerank = _config.enable_rerank
        if request.enable_rerank is not None:
            _config.enable_rerank = request.enable_rerank

        results = _retriever.retrieve(request.query)
        _config.enable_rerank = original_rerank

        # Filter by score threshold
        filtered_results = [r for r in results if r["score"] >= min_score]

        # Transform to response format
        doc_results = [
            DocumentResult(
                id=r["id"],
                text=r["text"],
                source=r["metadata"]["source"],
                score=float(r["score"]),
            )
            for r in filtered_results
        ]

        return RetrievalResponse(
            query=request.query, results=doc_results, total_results=len(doc_results)
        )

    except Exception as e:
        raise HTTPException(
            status_code=500,
            detail=f"Filtered retrieval failed: {str(e)}",
        )


@app.get("/config", tags=["Configuration"])
async def get_config() -> dict:
    """Get current retriever configuration.

    Returns:
        Dictionary with current hybrid retriever configuration parameters.
    """
    if _config is None:
        raise HTTPException(
            status_code=503,
            detail="Retriever not initialized",
        )

    return {
        "semantic_top_k": _config.semantic_top_k,
        "keyword_top_k": _config.keyword_top_k,
        "final_top_k": _config.final_top_k,
        "semantic_weight": _config.semantic_weight,
        "keyword_weight": _config.keyword_weight,
        "enable_rerank": _config.enable_rerank,
        "pre_rerank_top_k": _config.pre_rerank_top_k,
    }


if __name__ == "__main__":
    import uvicorn

    # Run the API server
    # Access at http://localhost:8000/docs for interactive API documentation
    print("\n🚀 Starting Hybrid RAG Retriever API...")
    print("📖 Swagger UI: http://localhost:8000/docs")
    print("📋 ReDoc: http://localhost:8000/redoc")
    print("🔧 Health Check: http://localhost:8000/health\n")
    
    uvicorn.run(app, host="0.0.0.0", port=8000)
